Временной ряд (По lat и long Сидней, также timezone тоже Сидней) wheather data от Historical Forecast API 

In [3]:
from dotenv import load_dotenv

In [3]:

import pandas as pd
import requests
import json

url = 'https://historical-forecast-api.open-meteo.com/v1/forecast?latitude=-33.86&longitude=151.21&start_date=2017-01-02&end_date=2026-09-13&daily=temperature_2m_mean&timezone=Australia%2FSydney'

response = requests.get(url)
response.raise_for_status()
data = response.json()

with open("weather_raw.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

df = pd.DataFrame(data['daily'])
df['time'] = pd.to_datetime(df['time'])
df.head()

df.to_csv('Sydney_mean_temp_decade.csv', index = False)


In [4]:
print(type(df.time[1]))

<class 'pandas.Timestamp'>


Многомерные данные: Данные о 3-х футбольных лигах за сезон 2023 года. PL - Премьер Лига, PD - Ла Лига, BL1 - Бундеслига с 8 параметрами 

In [5]:
import time
import os
API_football = os.getenv("API_KEY")

HEADERS = {"X-Auth-Token": API_football}

competitions = ["PL", "PD", "BL1"]

matches_data = []

for comp in competitions:
    url = f"https://api.football-data.org/v4/competitions/{comp}/matches?season=2023"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()

    with open(f"football_{comp}_league.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    matches = response.json().get("matches", [])

    for match in matches:
        matches_data.append(
            {
                "match_id": match.get("id"),
                "utc_date": match.get("utcDate"),
                "status": match.get("status"),
                "competition": match.get("competition", {}).get("name"),
                "matchday": match.get("matchday"),
                "home_team": match.get("homeTeam", {}).get("name"),
                "away_team": match.get("awayTeam", {}).get("name"),    
                "home_goals": match.get("score", {}).get("fullTime", {}).get("home"),
                "away_goals": match.get("score", {}).get("fullTime", {}).get("away"),
                "winner": match.get("score", {}).get("winner") 
            }
        )
    time.sleep(6)

df_football = pd.DataFrame(matches_data)
df_football.head()

df_football['utc_date'] = pd.to_datetime(df_football['utc_date'])
df_football.to_csv("football_3_ligues.csv", index=False)

In [6]:
print(df_football.duplicated().sum())
print(df_football.isnull().sum().sum())

0
0


In [7]:
df_football.head()

,match_id,utc_date,status,competition,matchday,home_team,away_team,home_goals,away_goals,winner
0,435943,2023-08-11 19:00:00+00:00,FINISHED,Premier League,1,Burnley FC,Manchester City FC,0,3,AWAY_TEAM
1,435944,2023-08-12 12:00:00+00:00,FINISHED,Premier League,1,Arsenal FC,Nottingham Forest FC,2,1,HOME_TEAM
2,435945,2023-08-12 14:00:00+00:00,FINISHED,Premier League,1,AFC Bournemouth,West Ham United FC,1,1,DRAW
3,435946,2023-08-12 14:00:00+00:00,FINISHED,Premier League,1,Brighton & Hove Albion FC,Luton Town FC,4,1,HOME_TEAM
4,435947,2023-08-12 14:00:00+00:00,FINISHED,Premier League,1,Everton FC,Fulham FC,0,1,AWAY_TEAM


Текстовые данные

In [ ]:
import os
import requests
import trafilatura
from trafilatura.settings import use_config
import json

config = use_config()
config.set("DEFAULT", "DOWNLOAD_TIMEOUT", "5")

API_news = os.getenv("NEWS_API_KEY")

HEADERS = {'Authorization': API_news}

categories = ["general", "society", "science_technology", "politics_government", "economy_business_finance", "arts_culture_entertainment", "lifestyle_leisure", "human_interest", "sport", "environment", "education", "labour", "health", "automotive"]
articles = []
seen_titles = set()
for category in categories:
  for page_number in range(1,4):

    response = requests.get(
      "https://api.currentsapi.services/v2/latest-news",
      headers=HEADERS,
      params={"page_number": page_number, "category": category},
    )
    response.raise_for_status()

    data = response.json()

    news = response.json().get("news", [])
    
    for novel in news:
        title = novel.get('title')

        if title and title in seen_titles:
            continue

        seen_titles.add(title)

        
        article_url = novel.get("url")
        downloaded = trafilatura.fetch_url(article_url)

        if not downloaded:
                continue
        
        full_text = trafilatura.extract(downloaded)

        articles.append({
            "title":novel.get("title"),
            "description":novel.get("description"),
            "author":novel.get("author"),
            "language":novel.get("language"),
            "published":novel.get("published"),
            "url": novel.get("url"),
            "full_text": full_text
        })
with open("news_text.json", "a", encoding="utf-8") as f:
  json.dump(articles, f, ensure_ascii=False, indent=4)
  f.write("\n")

In [ ]:
import os
import requests
import trafilatura
from trafilatura.settings import use_config
import json
from concurrent.futures import ThreadPoolExecutor  # 1. Добавили импорт

config = use_config()
config.set("DEFAULT", "DOWNLOAD_TIMEOUT", "5")

API_news = os.getenv("NEWS_API_KEY")
HEADERS = {'Authorization': API_news}

categories = ["general", "society", "science_technology", "politics_government", "economy_business_finance", "arts_culture_entertainment", "lifestyle_leisure", "human_interest", "sport", "environment", "education", "labour", "health", "automotive"]
articles = []
seen_titles = set()

def download_article(novel):
    article_url = novel.get("url")
    downloaded = trafilatura.fetch_url(article_url, config=config)

    if not downloaded:
        return None
    
    full_text = trafilatura.extract(downloaded)

    return {
        "title": novel.get("title"),
        "description": novel.get("description"),
        "author": novel.get("author"),
        "language": novel.get("language"),
        "published": novel.get("published"),
        "url": novel.get("url"),
        "full_text": full_text
    }

with ThreadPoolExecutor(max_workers=20) as executor:
    for category in categories:
        for page_number in range(1, 4):

            response = requests.get(
                "https://api.currentsapi.services/v2/latest-news",
                headers=HEADERS,
                params={"page_number": page_number, "category": category},
            )
            response.raise_for_status()

            data = response.json()
            news = response.json().get("news", [])
            
            for novel in news:
                title = novel.get('title')

                if not title or title in seen_titles:
                    continue

                seen_titles.add(title)

                future = executor.submit(download_article, novel)
                result = future.result()
                
                if result:
                    articles.append(result)

with open("news_text.json", "w", encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False, indent=4)

print(f"Готово! Сохранено статей: {len(articles)}")

KeyboardInterrupt: 

In [4]:
import os
import requests
import trafilatura
from trafilatura.settings import use_config
import json
from concurrent.futures import ThreadPoolExecutor, wait

# 1. Настройка таймаута trafilatura
config = use_config()
config.set("DEFAULT", "DOWNLOAD_TIMEOUT", "5")

API_news = os.getenv("NEWS_API_KEY")
HEADERS = {'Authorization': API_news}

categories = [
    "general", "society", "science_technology", "politics_government",
    "economy_business_finance", "arts_culture_entertainment", "lifestyle_leisure",
    "human_interest", "sport", "environment", "education", "labour", "health", "automotive"
]

articles = []
seen_titles = set()
futures_map = {}

def download_article(novel):
    article_url = novel.get("url")
    downloaded = trafilatura.fetch_url(article_url, config=config)

    if not downloaded:
        return None
    
    full_text = trafilatura.extract(downloaded)
    if not full_text:
        return None

    return {
        "title": novel.get("title"),
        "description": novel.get("description"),
        "author": novel.get("author"),
        "language": novel.get("language"),
        "published": novel.get("published"),
        "url": article_url,
        "full_text": full_text
    }

# 2. Создаем пул потоков
executor = ThreadPoolExecutor(max_workers=20)

try:
    for category in categories:
        for page_number in range(1, 4):
            response = requests.get(
                "https://api.currentsapi.services/v2/latest-news",
                headers=HEADERS,
                params={"page_number": page_number, "category": category},
            )
            response.raise_for_status()

            data = response.json()
            news = data.get("news", [])
            
            for novel in news:
                title = novel.get('title')

                # Исключаем пустые заголовки и дубликаты
                if not title or title in seen_titles:
                    continue

                seen_titles.add(title)

                # Отправляем задачу в поток
                future = executor.submit(download_article, novel)
                futures_map[future] = title

    print(f"{len(futures_map)}")

    # 3. Ждем максимум 20 секунд на скачивание ВСЕХ статей
    done, not_done = wait(futures_map.keys(), timeout=300)

    # Забираем только то, что успело скачаться
    for future in done:
        try:
            result = future.result()
            if result:
                articles.append(result)
        except Exception:
            pass

    print(f"success {len(articles)} drpop {len(not_done)}.")

finally:
    # 4. Принудительно отменяем все висячие потоки и закрываем пул
    executor.shutdown(wait=False, cancel_futures=True)

# 5. Сохраняем в валидный JSON-массив
with open("news_text.json", "w", encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False, indent=4)

HTTPError: 429 Client Error: Too Many Requests for url: https://api.currentsapi.services/v2/latest-news?page_number=1&category=general